In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
seed = 1
df = sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [ ]:
def ols(X, y):
  return np.linalg.inv(X.T @ X) @ X.T @ y

def mse(y, pred):
  return np.mean((y - pred) ** 2)

def rmse(y, pred):
  return np.sqrt(mse(y, pred))

def mae(y, pred):
  return np.mean(np.abs(y - pred))

def r_squared(y, pred):
  mean_y = np.mean(y)
  return 1 - (np.sum((y - pred) ** 2) / np.sum((y - mean_y) ** 2))

def adjusted_r_squared(y, pred, n_features):
  return 1 - (
    ((1 - r_squared(y, pred)) * (y.shape[0] - 1)) / 
    (y.shape[0] - n_features - 1)
  )


In [5]:
df['sex'] = df['sex'].cat.codes
df['smoker'] = df['smoker'].cat.codes
df['time'] = df['time'].cat.codes

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def produce_training_data(df, features):
  X = df[features]
  y = df['tip']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

  scaler = StandardScaler()
  X_train_standard = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  assert X_train_standard.shape == X_train.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_test_scaled.shape == X_test.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_train.shape[0] == y_train.shape[0], "X train's shape does not match the y train's shape"
  assert X_test.shape[0] == y_test.shape[0], "X test's shape does not match the y test's shape"
  assert X_train.shape[1] == X.shape[1] == X_test.shape[1], "Features shape does not match in X_train, X_test, and original X"


  X_train_standard = np.concat((np.ones(X_train_standard.shape[0]).reshape(-1, 1), X_train_standard), axis=1)
  X_test_scaled = np.concat((np.ones(X_test_scaled.shape[0]).reshape(-1, 1), X_test_scaled), axis=1)

  return X_train_standard, X_test_scaled, y_train.to_numpy(), y_test.to_numpy()


In [7]:
# Feature selection with Lasso regression

features = ['total_bill', 'size', 'sex', 'smoker', 'time']
X_train, X_test, y_train, y_test = produce_training_data(df, features)

weight_names = ['B' , *features]

In [11]:
features = []
results = []


for feature in ['total_bill', 'size', 'sex', 'smoker', 'time']:
  features.append(feature)

  X_train, X_test, y_train, y_test = produce_training_data(df, features)

  res_ols = ols(X_train, y_train)
  pred_ols = X_test @ res_ols

  results.append({
    "MSE": mse(y_test, pred_ols),
    "RMSE": rmse(y_test, pred_ols), 
    "MAE": mae(y_test, pred_ols), 
    "R2": r_squared(y_test, pred_ols), 
    "Adjusted_R2": adjusted_r_squared(y_test, pred_ols, X_test.shape[1]),
    "Features": " | ".join(features),
    "Algorithm": "OLS"
  })


pd.DataFrame(results)

,MSE,RMSE,MAE,R2,Adjusted_R2,Features,Algorithm
0,1.229862,1.108991,0.767950,0.532681,0.512363,total_bill,OLS
1,1.295447,1.138177,0.788599,0.507760,0.474944,total_bill | size,OLS
2,1.299680,1.140035,0.787776,0.506152,0.461256,total_bill | size | sex,OLS
3,1.309712,1.144427,0.765847,0.502340,0.444472,total_bill | size | sex | smoker,OLS
4,1.323246,1.150324,0.769956,0.497197,0.425368,total_bill | size | sex | smoker | time,OLS


In [9]:
# residual = (y_test - pred).to_numpy()

# sns.kdeplot(x=residual)
